In [1]:
import numpy as np

def parse_sign_matrix(text):
    """conver + to +1, - to -1"""
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    return np.array([[1 if c == '+' else -1 for c in line] for line in lines], dtype=int)

def normalize_pm_matrix(H):
    """normalize ±1 matrix s.t. first row and first column are all +1"""
    H = H.copy()
    n = len(H)
    for j in range(n):
        if H[0, j] == -1: H[:, j] *= -1
    for i in range(n):
        if H[i, 0] == -1: H[i, :] *= -1
    return H

def reduce_to_01_matrix(text):
    """conver NxN ±1 matrix to (N-1)x(N-1) 0/1 matrix"""
    H = normalize_pm_matrix(parse_sign_matrix(text))
    B = H[1:, 1:]
    A = (1 - B) // 2   # +1→0, -1→1
    return A


text = """
+++++++++++++++++
+----------++++++
+------++++----++
+++----+--++--+--
++--+---+-+-++---
+--+-+---++++----
+-+---+-++--+-+--
+---+-++-+-+-+---
+--+-+-++----++--
+---+++---+---+-+
+-++--+---+--+-+-
+++--+---+---+--+
++---+++----+--+-
++-+--+-+--+----+
++-++----+----++-
+-+++--+----+---+
+-+-++--+--+---+-
"""
A = reduce_to_01_matrix(text)
print("0/1 matrix:\n", A)
# det of A
print(abs(np.linalg.det(A)))

formatted = "upper_bound_matrix = [\n    " + ";\n    ".join(
    [" ".join(map(str, row)) for row in A]
) + "\n];"

print(formatted)

0/1 matrix:
 [[1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0]
 [1 1 1 1 1 1 0 0 0 0 1 1 1 1 0 0]
 [0 0 1 1 1 1 0 1 1 0 0 1 1 0 1 1]
 [0 1 1 0 1 1 1 0 1 0 1 0 0 1 1 1]
 [1 1 0 1 0 1 1 1 0 0 0 0 1 1 1 1]
 [1 0 1 1 1 0 1 0 0 1 1 0 1 0 1 1]
 [1 1 1 0 1 0 0 1 0 1 0 1 0 1 1 1]
 [1 1 0 1 0 1 0 0 1 1 1 1 0 0 1 1]
 [1 1 1 0 0 0 1 1 1 0 1 1 1 0 1 0]
 [1 0 0 1 1 0 1 1 1 0 1 1 0 1 0 1]
 [0 0 1 1 0 1 1 1 0 1 1 1 0 1 1 0]
 [0 1 1 1 0 0 0 1 1 1 1 0 1 1 0 1]
 [0 1 0 1 1 0 1 0 1 1 0 1 1 1 1 0]
 [0 1 0 0 1 1 1 1 0 1 1 1 1 0 0 1]
 [1 0 0 0 1 1 0 1 1 1 1 0 1 1 1 0]
 [1 0 1 0 0 1 1 0 1 1 0 1 1 1 0 1]]
327680.00000000023
upper_bound_matrix = [
    1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0;
    1 1 1 1 1 1 0 0 0 0 1 1 1 1 0 0;
    0 0 1 1 1 1 0 1 1 0 0 1 1 0 1 1;
    0 1 1 0 1 1 1 0 1 0 1 0 0 1 1 1;
    1 1 0 1 0 1 1 1 0 0 0 0 1 1 1 1;
    1 0 1 1 1 0 1 0 0 1 1 0 1 0 1 1;
    1 1 1 0 1 0 0 1 0 1 0 1 0 1 1 1;
    1 1 0 1 0 1 0 0 1 1 1 1 0 0 1 1;
    1 1 1 0 0 0 1 1 1 0 1 1 1 0 1 0;
    1 0 0 1 1 0 1 1 1 0 1 1 0 1 0 1;
    0 0 1 1 0 

In [11]:
# each row is a matrix, reshape it to 11 x 11 matrix
A = "01111011100,11011000111,00001111101,00111110110,11011110000,01101100001,10110101101,11001101110,10101011011,11100010101,01010111011"
A = np.array([[int(c) for c in row] for row in A.split(",")], dtype=int)
print("0/1 matrix:\n", A)

# calculate det
print(abs(np.linalg.det(A)))

0/1 matrix:
 [[0 1 1 1 1 0 1 1 1 0 0]
 [1 1 0 1 1 0 0 0 1 1 1]
 [0 0 0 0 1 1 1 1 1 0 1]
 [0 0 1 1 1 1 1 0 1 1 0]
 [1 1 0 1 1 1 1 0 0 0 0]
 [0 1 1 0 1 1 0 0 0 0 1]
 [1 0 1 1 0 1 0 1 1 0 1]
 [1 1 0 0 1 1 0 1 1 1 0]
 [1 0 1 0 1 0 1 1 0 1 1]
 [1 1 1 0 0 0 1 0 1 0 1]
 [0 1 0 1 0 1 1 1 0 1 1]]
765.0000000000007


In [16]:
# Modified version to store matrices for each determinant value
from collections import Counter, defaultdict

N = 17

def decode_matrix(line):
    """decode a line of 0/1 string to a matrix"""
    try: 
        row_strs = line.strip().split(",")

        if len(row_strs) == N:
            matrix = np.array([[int(c) for c in row] for row in row_strs], dtype=int)
        else:
            return None
        
        if matrix.shape != (N, N):
            return None  
        return matrix
    
    except Exception:
        return None

def canonical_form(A):
    """return the canonical form of matrix A under row and column permutations"""
    A_sorted_rows = np.array(sorted(A.tolist()))
    A_sorted_cols = np.array(sorted(A_sorted_rows.T.tolist())).T
    return ''.join(map(str, A_sorted_cols.flatten().tolist()))

file_path = "C:/Users/123li/Downloads/Project/gpu_run_output/dim17_run_5/1111/transformer-output-decoded.txt"
with open(file_path, encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]

invalid_count = 0
canonical_strings = {}
det_to_matrices = defaultdict(list)  # Store matrices for each determinant

for line in lines:
    A = decode_matrix(line)
    if A is None:
        invalid_count += 1
        continue
    
    can_str = canonical_form(A)
    if can_str not in canonical_strings:  # ensure no repeats
        det = round(abs(np.linalg.det(A))) 
        canonical_strings[can_str] = det
        # Store the matrix for this determinant
        det_to_matrices[det].append(A.copy())

print(f"Total matrices: {len(lines)}")
print(f"Unique (up to permutation): {len(canonical_strings)}")
print(f"Duplicate rate: {(1 - len(canonical_strings)/len(lines))*100:.2f}%")
print(f"Invalid matrices: {invalid_count}")
print("\n")

# Determinant distribution with matrices
det_counts = Counter(canonical_strings.values())
print("Det distribution with sample matrices:")
print("=" * 50)

sort_dets = sorted(det_counts.items(), reverse=True)
for rank, (det, count) in enumerate(sort_dets, start=1):
    print(f"\nDeterminant: {det}, Count: {count}")
    
    # Print up to 3 example matrices for this determinant
    if rank <= 5:
        matrices_for_det = det_to_matrices[det]
        num_to_show = min(2, len(matrices_for_det))
        
        for i in range(num_to_show):
            print(f"Example matrix {i+1}:")
            print(matrices_for_det[i])
            if i < num_to_show - 1:
                print()
        
        if len(matrices_for_det) > num_to_show:
            print(f"... and {len(matrices_for_det) - num_to_show} more matrices with det = {det}")
    print("=" * 50)

Total matrices: 30100
Unique (up to permutation): 180
Duplicate rate: 99.40%
Invalid matrices: 837


Det distribution with sample matrices:

Determinant: 1114112, Count: 1
Example matrix 1:
[[1 0 1 1 1 1 1 1 0 0 1 1 0 0 1 0 0]
 [0 0 0 1 1 1 1 1 1 1 0 1 0 1 0 0 1]
 [0 1 0 0 1 1 1 1 0 0 1 0 0 1 1 1 1]
 [0 1 1 0 0 1 1 1 0 1 0 1 1 1 1 0 0]
 [0 1 1 1 0 0 1 1 1 1 1 0 0 0 1 0 1]
 [0 1 1 1 1 0 0 1 0 0 1 1 1 1 0 0 1]
 [0 1 1 1 1 1 0 0 0 1 0 1 0 0 1 1 1]
 [1 0 0 0 0 0 0 1 1 0 0 1 1 0 1 1 1]
 [1 1 0 0 1 0 0 0 1 1 1 1 0 1 1 0 0]
 [1 1 0 1 1 0 1 1 0 1 0 0 1 0 0 1 0]
 [0 0 1 0 1 1 0 1 1 1 1 0 1 0 0 1 0]
 [0 1 0 1 0 1 1 0 1 0 1 1 1 0 0 1 0]
 [1 0 0 1 0 1 0 0 0 1 1 0 1 1 1 0 1]
 [1 1 1 1 0 1 0 1 1 0 0 0 0 1 0 1 0]
 [0 0 1 1 1 0 1 0 1 0 0 0 1 1 1 1 0]
 [1 0 1 0 0 0 1 0 0 1 1 1 0 1 0 1 1]
 [1 1 1 0 1 1 1 0 1 0 0 0 1 0 0 0 1]]

Determinant: 999424, Count: 1
Example matrix 1:
[[0 1 1 0 1 0 1 1 1 0 1 0 0 0 1 1 0]
 [0 1 1 0 1 1 0 0 1 1 0 1 1 0 1 0 1]
 [0 1 1 1 0 1 1 1 0 1 1 0 1 0 0 0 1]
 [1 1 0 1 0 1 0 1 1 

In [ ]:
# Interactive functions to explore matrices by determinant value

def show_matrices_with_det(target_det, max_show=5):
    """Show all matrices with a specific determinant value"""
    if target_det not in det_to_matrices:
        print(f"No matrices found with determinant {target_det}")
        return
    
    matrices = det_to_matrices[target_det]
    print(f"Found {len(matrices)} unique matrices with determinant {target_det}")
    print("=" * 60)
    
    num_to_show = min(max_show, len(matrices))
    for i in range(num_to_show):
        print(f"\nMatrix {i+1}/{len(matrices)}:")
        print(matrices[i])
        
        # Also show some properties
        print(f"Shape: {matrices[i].shape}")
        print(f"Number of 1s: {np.sum(matrices[i])}")
        print(f"Actual determinant: {np.linalg.det(matrices[i]):.6f}")
    
    if len(matrices) > max_show:
        print(f"\n... and {len(matrices) - max_show} more matrices not shown")

def compare_matrices_same_det(target_det):
    """Compare first two matrices with the same determinant"""
    if target_det not in det_to_matrices or len(det_to_matrices[target_det]) < 2:
        print(f"Need at least 2 matrices with determinant {target_det}")
        return
    
    matrices = det_to_matrices[target_det]
    A, B = matrices[0], matrices[1]
    
    print(f"Comparing two matrices with determinant {target_det}:")
    print("\nMatrix A:")
    print(A)
    print("\nMatrix B:")
    print(B)
    
    print(f"\nAre they identical? {np.array_equal(A, B)}")
    print(f"Same number of 1s? {np.sum(A) == np.sum(B)}")
    
    # Check if they're equivalent via your canonical form
    can_A = canonical_form(A)
    can_B = canonical_form(B)
    print(f"Same canonical form? {can_A == can_B}")

# Example usage
print("Available determinant values:", sorted(det_to_matrices.keys(), reverse=True))
print("\nUse show_matrices_with_det(det_value) to see matrices with specific determinant")
print("Use compare_matrices_same_det(det_value) to compare matrices with same determinant")

In [ ]:
import networkx as nx
import numpy as np

def equivalent_via_graph(A, B):
    A = np.array(A)
    B = np.array(B)
    n, m = A.shape
    if B.shape != (n, m):
        return False

    G1 = nx.Graph()
    G2 = nx.Graph()

    # Add bipartite nodes: rows and columns
    G1.add_nodes_from(range(n), bipartite=0)
    G1.add_nodes_from(range(n, n+m), bipartite=1)
    G2.add_nodes_from(range(n), bipartite=0)
    G2.add_nodes_from(range(n, n+m), bipartite=1)

    # Add edges for 1 entries
    for i in range(n):
        for j in range(m):
            if A[i, j]:
                G1.add_edge(i, n + j)
            if B[i, j]:
                G2.add_edge(i, n + j)

    # Check isomorphism
    return nx.is_isomorphic(G1, G2)

# Example with corrected syntax
A = [[1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1],
     [0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0],
     [1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0],
     [0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0],
     [1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1],
     [0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1],
     [1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0],
     [0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0],
     [1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1],
     [1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0],
     [1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0],
     [0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1],
     [0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1],
     [1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1]]

B = [[1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0],
     [1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0],
     [0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0],
     [0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1],
     [0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1],
     [1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1],
     [0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
     [1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0], 
     [0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0],
     [1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1],
     [1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0],
     [1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1],
     [1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0],
     [0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1]]

C = np.array([
    [0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0],
    [1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1],
    [1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1],
    [1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1],
    [0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1],
    [0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0],
    [0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1],
    [0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1],
    [0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1],
    [1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0],
    [1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0],
    [1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0],
    [1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1],
    [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0]
], dtype=int)

print(np.linalg.det(A))
print(np.linalg.det(B))
print(np.linalg.det(C))

print(equivalent_via_graph(A, B))
print(equivalent_via_graph(A, C))
print(equivalent_via_graph(B, C))

-25515.000000000033
-25514.999999999985
-25514.999999999985
False
False
False


In [ ]:
# Different ways to format matrices with commas

# Method 1: Convert your existing matrix to comma-separated format
A_original = [[1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1],
              [0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0],
              [1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0],
              [0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0],
              [1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1],
              [0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1],
              [1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0],
              [0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0],
              [1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1],
              [1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0],
              [1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0],
              [0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1],
              [0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1],
              [1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1]]

# Method 2: Convert matrix to your file format (comma-separated rows)
def matrix_to_comma_string(matrix):
    """Convert matrix to comma-separated string format like your file"""
    return ",".join(["".join(map(str, row)) for row in matrix])

comma_string = matrix_to_comma_string(A_original)
print("Comma-separated format for file:")
print(comma_string)
print()

# Method 3: Pretty print with commas
def print_matrix_with_commas(matrix, name="Matrix"):
    """Print matrix in a readable format with commas"""
    print(f"{name} = [")
    for i, row in enumerate(matrix):
        if i == len(matrix) - 1:  # Last row
            print(f"    {row}]")
        else:
            print(f"    {row},")

print_matrix_with_commas(A_original, "A")
print()

# Method 4: Convert numpy array to comma format
A_numpy = np.array(A_original)
print("NumPy array:")
print(A_numpy)
print(f"Shape: {A_numpy.shape}")
print(f"Determinant: {np.linalg.det(A_numpy):.6f}")

# Method 5: One-liner to add commas to your matrix definition
def format_matrix_code(matrix):
    """Generate Python code with proper comma formatting"""
    lines = ["A = ["]
    for i, row in enumerate(matrix):
        comma = "," if i < len(matrix) - 1 else ""
        lines.append(f"     {row}{comma}")
    lines.append("]")
    return "\n".join(lines)

print("\nFormatted Python code:")
print(format_matrix_code(A_original))

In [ ]:
from collections import Counter

N = 14

def decode_matrix(line):
    """decode a line of 0/1 string to a matrix"""
    try: 
        row_strs = line.strip().split(",")
        if len(row_strs) == N:
            # each element is a row string
            # must have N elements
            # if valid, convert to a matrix form
            matrix = np.array([[int(c) for c in row] for row in row_strs], dtype=int)
        else:
            return None
        
        if matrix.shape != (N, N):
            return None  
        return matrix
    
    except:
        return None

def matrix_to_bipartite_graph(A):
    """Convert binary matrix to a labeled bipartite graph."""
    A = np.array(A, dtype=int)
    n, m = A.shape
    G = nx.Graph()

    # Add labeled nodes to distinguish row vs column partitions
    for i in range(n):
        G.add_node(f"r{i}", bipartite=0, label="row")
    for j in range(m):
        G.add_node(f"c{j}", bipartite=1, label="col")

    # Add edges for 1 entries
    for i in range(n):
        for j in range(m):
            if A[i, j]:
                G.add_edge(f"r{i}", f"c{j}")
    return G, n, m

def canonical_form(A):
    """Return canonical form of a binary matrix under row/column permutations."""
    G, n, m = matrix_to_bipartite_graph(A)

    # Compute canonical labeling using Bliss algorithm
    G_can = nx.algorithms.isomorphism.canonical_form(G, node_attr='label')

    # Extract node order: rows first, then columns (as relabeled)
    row_nodes = [n for n, d in G_can.nodes(data=True) if d['label'] == 'row']
    col_nodes = [n for n, d in G_can.nodes(data=True) if d['label'] == 'col']

    # Convert to adjacency matrix
    adj = nx.to_numpy_array(G_can, nodelist=row_nodes + col_nodes, dtype=int)

    # Extract top-right block (rows × cols)
    return adj[:len(row_nodes), len(row_nodes):].astype(int)

file_path = "C:/Users/123li/Downloads/Project/gpu_run_output/dim14_run_1/1111/search_output_3.txt"
with open(file_path, encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]

invalid_count = 0
canonical_strings = {}
for line in lines:
    A = decode_matrix(line)
    if A is None:
        invalid_count += 1
        continue
    can_str = canonical_form(A)
    if can_str not in canonical_strings: # ensure no repeats
        # get the distribution of det in these unique matrices
        det = round(abs(np.linalg.det(A))) 
        canonical_strings[can_str] = det

print(f"Total matrices: {len(lines)}")
print(f"Unique (up to permutation): {len(canonical_strings)}")
print(f"Duplicate rate: {(1 - len(canonical_strings)/len(lines))*100:.2f}%")
print(f"Invalid matrices: {invalid_count}")
print("\n")

# unique det distribution
det_counts = Counter(canonical_strings.values())
print("Det distribution:")
for det, count in sorted(det_counts.items(), reverse=True):
    print(f"Det: {det}, Count: {count}")

AttributeError: module 'networkx.algorithms.isomorphism' has no attribute 'canonical_form'